In [0]:
use catalog smart_odoo;

CREATE TABLE IF NOT EXISTS silver.res_partner
(
    partner_id int PRIMARY KEY,
    partner_name string,
    company_id int,
    company_name STRING,
    parent_id int,
    parent_name string,
    commercial_partner_id int,
    commercial_partner_name STRING,
    country_id int,
    country_name STRING,
    state_id int,
    state_name STRING,
    industry_id int,
    is_company boolean,
    active boolean,
    customer_rank integer,
    supplier_rank integer,
    email string,
    phone string,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at       TIMESTAMP,
    dwh_source_db string
);

MERGE INTO silver.res_partner AS target
USING (
    SELECT
        cast( rp.id As int)   AS partner_id,
        rp.name  AS partner_name,

        cast( GET_JSON_OBJECT(rp.company_id ,'$[0]') As int)     AS company_id,
        GET_JSON_OBJECT(rp.company_id ,'$[1]')                   AS company_name,

        cast( GET_JSON_OBJECT(rp.parent_id ,'$[0]') As int)     AS parent_id,
        GET_JSON_OBJECT(rp.parent_id ,'$[1]')                   AS parent_name,

        cast( GET_JSON_OBJECT(rp.commercial_partner_id ,'$[0]') As int)       AS commercial_partner_id,
        GET_JSON_OBJECT(rp.commercial_partner_id ,'$[1]')       AS commercial_partner_name,

        cast( GET_JSON_OBJECT(rp.country_id ,'$[0]') As int)  AS country_id,
        GET_JSON_OBJECT(rp.country_id ,'$[1]')            AS country_name,

        cast( GET_JSON_OBJECT(rp.state_id ,'$[0]') As int) AS state_id,
        get_json_object(rp.state_id ,'$[1]')               AS state_name,

        cast( rp.industry_id As int)                AS industry_id,
        CAST(rp.is_company AS boolean)  AS is_company,
        CAST(rp.active AS boolean)      AS active,
        CAST(rp.customer_rank AS int)   AS customer_rank,
        CAST(rp.supplier_rank AS int)   AS supplier_rank,
        rp.email                        AS email,
        rp.phone                        AS phone,
        CAST(rp.create_date AS TIMESTAMP) AS created_at,
        CAST(rp.write_date  AS TIMESTAMP) AS updated_at,
        current_timestamp()            AS dwh_loaded_at,
        rp.dwh_source_db               AS dwh_source_db
    FROM bronze.res_partner rp
    WHERE CAST(rp.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date
         FROM silver.load_tracking
         WHERE table_name = 'res_partner'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.partner_id = source.partner_id

WHEN MATCHED THEN UPDATE SET
    target.partner_name = source.partner_name,
    target.company_id = source.company_id,
    target.company_name = source.company_name,
    target.parent_id = source.parent_id,
    target.parent_name = source.parent_name,
    target.commercial_partner_id = source.commercial_partner_id,
    target.commercial_partner_name = source.commercial_partner_name,
    target.country_id = source.country_id,
    target.country_name = source.country_name,
    target.state_id = source.state_id,
    target.state_name = source.state_name,
    target.industry_id = source.industry_id,
    target.is_company = source.is_company,
    target.active = source.active,
    target.customer_rank = source.customer_rank,
    target.supplier_rank = source.supplier_rank,
    target.email = source.email,
    target.phone = source.phone,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at,
    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *
;


In [0]:
USE CATALOG smart_odoo;

CREATE TABLE IF NOT EXISTS silver.res_users
(
    user_id int,

    company_id int,
    company_name STRING,

    partner_id int,
    partner_name STRING,

    sale_team_id int,
    sale_team_name STRING,

    login STRING,
    active BOOLEAN,
    share BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
);

MERGE INTO silver.res_users AS target
USING (
    SELECT
        CAST(ru.id AS int)                AS user_id,
        cast( GET_JSON_OBJECT(ru.company_id ,'$[0]') As int)                   AS company_id,
        GET_JSON_OBJECT(ru.company_id ,'$[1]')                   AS company_name,

        cast( GET_JSON_OBJECT(ru.partner_id ,'$[0]') As int)                  AS partner_id,
        GET_JSON_OBJECT(ru.partner_id ,'$[1]')                   AS partner_name,

        cast( GET_JSON_OBJECT(ru.sale_team_id ,'$[0]')  As int)     AS sale_team_id,
        GET_JSON_OBJECT(ru.sale_team_id ,'$[1]')     AS sale_team_name,
        ru.login                             AS login,
        CAST(ru.active AS BOOLEAN)           AS active,
        CAST(ru.share AS BOOLEAN)            AS share,
        CAST(ru.create_date AS TIMESTAMP)    AS created_at,
        CAST(ru.write_date AS TIMESTAMP)     AS updated_at,
        current_timestamp()                  AS dwh_loaded_at,
        ru.dwh_source_db                               AS dwh_source_db
    FROM bronze.res_users ru
    WHERE CAST(ru.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date
         FROM silver.load_tracking
         WHERE table_name = 'res_users'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.user_id = source.user_id

WHEN MATCHED THEN UPDATE SET
    target.company_id = source.company_id,
    target.company_name = source.company_name,
    target.partner_id = source.partner_id,
    target.partner_name = source.partner_name,
    target.sale_team_id = source.sale_team_id,
    target.sale_team_name = source.sale_team_name,
    target.login = source.login,
    target.active = source.active,
    target.share = source.share,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at,
    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;

In [0]:
use catalog smart_odoo;

CREATE TABLE IF NOT EXISTS silver.res_partner_category
(
    category_id int,
    parent_id int,
    parent_name string,
    parent_path STRING,
    category_name STRING,
    color INT,
    active BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
);

MERGE INTO silver.res_partner_category AS target
USING (
    SELECT
        CAST(rpc.id AS int)               AS category_id,
        cast( GET_JSON_OBJECT(rpc.parent_id ,'$[0]') As int)        AS parent_id,
        GET_JSON_OBJECT(rpc.parent_id ,'$[1]')        AS parent_name,
        rpc.parent_path                      AS parent_path,
        rpc.name                             AS category_name,
        CAST(rpc.color AS INT)               AS color,
        CAST(rpc.active AS BOOLEAN)          AS active,
        CAST(rpc.create_date AS TIMESTAMP)   AS created_at,
        CAST(rpc.write_date AS TIMESTAMP)    AS updated_at,
        current_timestamp()                  AS dwh_loaded_at,
        rpc.dwh_source_db                    AS dwh_source_db
    FROM bronze.res_partner_category rpc
    WHERE CAST(rpc.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date
         FROM silver.load_tracking
         WHERE table_name = 'res_partner_category'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.category_id = source.category_id

WHEN MATCHED THEN UPDATE SET
    target.parent_id = source.parent_id,
    target.parent_name = source.parent_name,
    target.parent_path = source.parent_path,
    target.category_name = source.category_name,
    target.color = source.color,
    target.active = source.active,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at,
    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;

In [0]:
use catalog smart_odoo;

CREATE TABLE IF NOT EXISTS silver.res_groups
(
    group_id int,
    group_name STRING,
    privilege_id int,
    privilege_name string,
    share BOOLEAN,
    group_comment STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP,
    dwh_source_db STRING
);

MERGE INTO silver.res_groups AS target
USING (
    SELECT
        CAST(rg.id AS int)               AS group_id,
        rg.name                             AS group_name,
        cast( get_json_object (rg.privilege_id ,'$[0]') As int)     AS privilege_id,
        GET_JSON_OBJECT(rg.privilege_id ,'$[1]')    AS privilege_name,
        CAST(rg.share AS BOOLEAN)           AS share,
        rg.comment                          AS group_comment,
        CAST(rg.create_date AS TIMESTAMP)   AS created_at,
        CAST(rg.write_date AS TIMESTAMP)    AS updated_at,
        current_timestamp()                 AS dwh_loaded_at,
        rg.dwh_source_db                    AS dwh_source_db
    FROM bronze.res_groups rg
    WHERE CAST(rg.write_date AS TIMESTAMP) > COALESCE(
        (SELECT last_write_date
         FROM silver.load_tracking
         WHERE table_name = 'res_groups'),
        CAST('1900-01-01' AS TIMESTAMP)
    )
) AS source
ON target.group_id = source.group_id

WHEN MATCHED THEN UPDATE SET
    target.group_name = source.group_name,
    target.privilege_id = source.privilege_id,
    target.privilege_name = source.privilege_name,
    target.share = source.share,
    target.group_comment = source.group_comment,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at,
    target.dwh_loaded_at = current_timestamp(),
    target.dwh_source_db = source.dwh_source_db

WHEN NOT MATCHED THEN INSERT *;